# SafeCrop · Day 2: make the app's model files (Google Colab, T4 GPU)

This notebook makes **every file the back end needs** to name a disease, in one run (about **25 minutes** on a T4):

| File | What it is |
| --- | --- |
| `clip_ft_delta.safetensors` | our fine-tuned CLIP ViT-L/14 (E4 settings, trained on all 240 photos): only the changed weights |
| `database.npz` | the picture database: 480 photos × (clean + 15 noisy copies) = 7,680 pictures, 768 numbers each |
| `thumbs/` + `photos.csv` | a 200-pixel picture of every clean photo (the app always shows the clean one), with source and licence |
| `photo_check.joblib` + `photo_check.py` | the photo check: 16 measurements + two random forests (problem and level) |
| `thresholds.json` | the cut-offs: "sure", "not sure", "not a leaf", and whether to use the resample fix |
| `README.md`, `export_info.json`, `checks/` | every number for the model card and slides, library versions, per-image results |

### Before you run (once)
1. In Google Drive make the folder **My Drive/SafeCrop_app** and upload **safecrop_app_inputs.zip** into it
   (made on the laptop with `python notebooks/make_colab_inputs.py`).
2. *Optional:* put 20–30 phone photos of things that are **not leaves** (desk, hand, sky, food, floor, wall) into
   **My Drive/SafeCrop_app/non_leaf/**. Without them the "not a leaf" cut-off uses only ~14 built-in sample pictures.
3. *Optional, for the Hugging Face upload:* click the **key icon** (Secrets) on the left → **Add new secret** →
   name `HF_TOKEN`, value = your *write* token → switch **Notebook access** on. The token is never printed or saved.

### Run
**Runtime → Change runtime type → T4 GPU → Save**, then **Runtime → Run all**. Allow Google Drive when asked.
Everything is saved in **My Drive/SafeCrop_app/app_model_files/** and **app_model_files.zip** (and, with the
token, in your private Hugging Face repo `safecrop-models`).

**Honest limit:** the app model trains on every photo, so it must **never** be scored on the 960 test images.
Accuracy for the report stays the 5-fold numbers of the research repo. The fair checks here use a separate fold-0 model.

In [ ]:
# ============================ SETTINGS (the only cell you may want to change) ============================
DRIVE_FOLDER = "SafeCrop_app"             # folder in My Drive that holds the inputs zip
INPUTS_ZIP = "safecrop_app_inputs.zip"    # made on the laptop: python notebooks/make_colab_inputs.py
NON_LEAF_DIR = "non_leaf"                 # optional sub-folder of DRIVE_FOLDER: ~30 photos that are NOT leaves
OUT_NAME = "app_model_files"              # the output folder (and zip) saved in DRIVE_FOLDER

SEED = 299                                # same seed as E3/E4 (fold k used 299 + k)
THRESHOLDS_FROM = "recipe_b"              # which out-of-fold E4 answers set the cut-offs: "recipe_b" (noise made
                                          # another way: closest to real photos), "test" (test images) or "both"
SURE_ACCURACY = 0.95                      # "sure": answers at this vote share are right at least 95% of the time
NOT_SURE_ACCURACY = 0.70                  # "not sure": below this cut-off answers are right less than 70% of the time
PHOTO_CHECK_SURE = 0.60                   # trust the photo check's forest only when it is at least 60% sure

UPLOAD_TO_HF = True                       # needs the Colab secret HF_TOKEN (write rights); skipped if it is missing
HF_REPO_NAME = "safecrop-models"          # private model repo: <your user name>/safecrop-models

SMOKE_TEST = False                        # True = tiny fake CLIP + 1 epoch: only for testing this notebook on a laptop
LOCAL_DRIVE = ""                          # only outside Colab: a folder that plays the role of "My Drive"
# =========================================================================================================

In [ ]:
# ---- Step 0: GPU, small extra packages, library versions ----
import importlib, importlib.util, os, platform, subprocess, sys, time

T_START = time.time()
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:                                   # no "google" package at all: not Colab
    IN_COLAB = False
REPORT, NUMBERS, TIMES, _now = [], {}, {}, {}


def start(name):
    """A clear header for every step, and its clock."""
    _now["name"], _now["t"] = name, time.time()
    print(f"\n━━━━━━━━━━ {name} ━━━━━━━━━━", flush=True)


def done(msg=""):
    sec = time.time() - _now["t"]
    TIMES[_now["name"]] = round(sec, 1)
    print(f"✔ {_now['name']}: done in {sec:.0f} s. {msg}", flush=True)


def md_table(df):
    """A small markdown table (for README.md) from a DataFrame."""
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)


def right(ok, sel):
    """'97.9% (47/48)': how often `ok` is True where `sel` is True."""
    ok, sel = np.asarray(ok, bool), np.asarray(sel, bool)
    n = int(sel.sum())
    return f"{ok[sel].mean():.1%} ({int(ok[sel].sum())}/{n})" if n else "-"


start("Step 0: GPU, packages, versions")
if importlib.util.find_spec("pillow_heif") is None:          # opens iPhone (HEIC) photos in non_leaf/
    print("installing pillow-heif ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pillow-heif"], check=True)

import cv2, joblib, PIL, pillow_heif, safetensors, scipy, sklearn, transformers, huggingface_hub
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from PIL import Image

pillow_heif.register_heif_opener()
transformers.logging.set_verbosity_error()                    # hides long "unused weights" warnings
torch.backends.cuda.matmul.allow_tf32 = False                 # exact float32 maths, like the CPU in the app
torch.backends.cudnn.allow_tf32 = False

if torch.cuda.is_available():
    DEVICE, GPU = torch.device("cuda"), torch.cuda.get_device_name(0)
elif SMOKE_TEST:
    DEVICE, GPU = torch.device("cpu"), "none (smoke test on CPU)"
else:
    raise RuntimeError("No GPU! Click Runtime -> Change runtime type -> T4 GPU -> Save, then Runtime -> Run all again.")

VERSIONS = {"python": platform.python_version(), "torch": torch.__version__, "cuda": str(torch.version.cuda),
            "gpu": GPU, "transformers": transformers.__version__, "safetensors": safetensors.__version__,
            "huggingface_hub": huggingface_hub.__version__, "scikit-learn": sklearn.__version__,
            "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
            "opencv": cv2.__version__, "pillow": PIL.__version__, "joblib": joblib.__version__}
for k, v in VERSIONS.items():
    print(f"  {k:16s} {v}")
done(f"GPU: {GPU}")

## Step 1 · Google Drive and the inputs zip
Connects Google Drive (click **Connect to Google Drive**, pick your account, **Continue / Allow**), finds
`safecrop_app_inputs.zip`, unpacks it, and prepares the output folder. An older output folder is renamed, never deleted.

In [ ]:
start("Step 1: Google Drive and the inputs zip")
import datetime, gc, json, random, shutil, zipfile
from pathlib import Path

DHAKA = datetime.timezone(datetime.timedelta(hours=6))                      # Colab's clock is UTC
STAMP = datetime.datetime.now(DHAKA).strftime("%Y%m%d-%H%M")                # Dhaka time, on every output
if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print("Could not connect Google Drive:", e)
    MY_DRIVE, WORK = Path("/content/drive/MyDrive"), Path("/content/safecrop_work")
else:                                                         # laptop test: a normal folder plays "My Drive"
    MY_DRIVE = Path(LOCAL_DRIVE or ".").resolve()
    WORK = MY_DRIVE.parent / "safecrop_work"
DRIVE_OK = MY_DRIVE.is_dir()
BASE = MY_DRIVE / DRIVE_FOLDER


def find_inputs_zip():
    """My Drive/<DRIVE_FOLDER>/<zip>, else 1-2 folders deep anywhere in My Drive, else dragged into /content."""
    if DRIVE_OK:
        for p in [BASE / INPUTS_ZIP, *sorted(MY_DRIVE.glob(f"*/{INPUTS_ZIP}")),
                  *sorted(MY_DRIVE.glob(f"*/*/{INPUTS_ZIP}"))]:
            if p.is_file():
                return p
    for p in [Path("/content") / INPUTS_ZIP, Path.cwd() / INPUTS_ZIP]:
        if p.is_file():
            return p
    return None


ZIP = find_inputs_zip()
if ZIP is None:
    raise FileNotFoundError(f"{INPUTS_ZIP} was not found. Upload it to My Drive/{DRIVE_FOLDER}/ (or drag it into the "
                            "Files panel on the left), then Runtime -> Run all again.")
if DRIVE_OK and MY_DRIVE in ZIP.parents:
    BASE = ZIP.parent                                         # the outputs go next to the zip
print(f"Inputs: {ZIP} ({ZIP.stat().st_size / 1e6:.0f} MB)")

RESEARCH = WORK / "safecrop_research"                         # the research repo's code + data, unpacked
marker, sig = WORK / "unzipped.txt", f"{ZIP.stat().st_size}-{int(ZIP.stat().st_mtime)}"
if not (marker.is_file() and marker.read_text() == sig and RESEARCH.is_dir()):
    shutil.rmtree(WORK, ignore_errors=True)
    WORK.mkdir(parents=True)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(WORK)
    marker.write_text(sig)
if str(RESEARCH / "src") not in sys.path:
    sys.path.insert(0, str(RESEARCH / "src"))
BUNDLE = json.loads((RESEARCH / "BUNDLE_INFO.json").read_text())
print(f"Research code: NabilRafi/SafeCrop commit {BUNDLE['research_commit'][:7]} (zip made {BUNDLE['made']})")

OUT = WORK / OUT_NAME                                         # written here first (fast local disk) ...
shutil.rmtree(OUT, ignore_errors=True)
(OUT / "thumbs").mkdir(parents=True)
(OUT / "checks").mkdir()
DRIVE_OUT = BASE / OUT_NAME if DRIVE_OK else None             # ... and copied to Google Drive after every block
if DRIVE_OUT is not None and DRIVE_OUT.exists():
    old = DRIVE_OUT.with_name(f"{OUT_NAME}_old_{datetime.datetime.now(DHAKA):%Y%m%d-%H%M%S}")
    DRIVE_OUT.rename(old)
    print(f"An earlier {OUT_NAME} folder in Drive was kept as {old.name} (nothing is deleted)")
_synced = {}


def save_to_drive(note=""):
    """Copies new or changed output files to Google Drive, so a cut-off session loses little."""
    if DRIVE_OUT is None:
        print(f"  (no Google Drive: the files stay in {OUT})")
        return
    n = 0
    for src in sorted(OUT.rglob("*")):
        if src.is_file():
            key = (src.stat().st_size, src.stat().st_mtime_ns)
            if _synced.get(src) != key:
                dst = DRIVE_OUT / src.relative_to(OUT)
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(src, dst)
                _synced[src] = key
                n += 1
    print(f"  saved {n} new or changed files to Google Drive: {DRIVE_OUT}" + (f" ({note})" if note else ""))


done(f"outputs go to {DRIVE_OUT or OUT}")

## Step 2 · Load the data and check it
The 960 test images (240 photos × L0–L3, 8 diseases) and the 240 PlantVillage extras. It also checks that this Colab
still rebuilds test images exactly from their clean photo, because the database's noisy copies use the same recipe.

In [ ]:
start("Step 2: load the data and check it")
from scipy.stats import binomtest
from safecrop.clip import MEAN as CLIP_MEAN, STD as CLIP_STD, TINY, embed_arrays, fit_size, load_clip
from safecrop.config import load_config
from safecrop.data import disease_list, fold_photos, image_path, load_manifest, photos_table, read_rgb
from safecrop.evaluate import database_vote
from safecrop.features import noisy_copies_table
from safecrop.finetune import train_model
from safecrop.noise import NOISE_TYPES, load_square, make_noisy, seed_for

cfg = load_config()                                   # the research repo's configs/default.yaml = the E3/E4 settings
cfg["black_gram"] = True                              # 8 diseases: 240 photos, 960 images
cfg["seed"] = SEED
cfg["finetune"]["db_noisy_copies"] = True             # E4 = E3 + fine-tuned noisy copies in the database
COPY_SEED = int(cfg["noisy_copies"]["seed"])          # 2026: never 299, the test set's seed family
K = int(cfg["database"]["k"])                         # the 5 most similar pictures vote
assert COPY_SEED != 299 and K == 5

m = load_manifest(cfg)                                # one row per test image
P = photos_table(m)                                   # one row per photo (its clean L0 image)
DISEASES = disease_list(m)
missing = [im for im in m.image if not image_path(cfg, im).exists()]
assert len(m) == 960 and len(P) == 240 and len(DISEASES) == 8 and not missing, f"{len(missing)} images missing"

labels = pd.read_csv(RESEARCH / "data" / "labels.csv").set_index("key")
ex = pd.read_csv(RESEARCH / "data" / "extra_references.csv").sort_values(["option", "path"]).reset_index(drop=True)
ex["photo_id"] = "pv_" + ex.option + "_" + (ex.groupby("option").cumcount() + 1).map("{:02d}".format)
ex["label"] = [o if o in DISEASES else ("healthy_" + o.split("_")[0] if o.endswith("_healthy") else "other_" + o)
               for o in ex.option]                    # PlantVillage apple scab stays "apple_scab" (a 2nd photo source)
ex["file"] = [RESEARCH / "data" / p for p in ex.path]
assert len(ex) == 240 and all(f.is_file() for f in ex.file)
print(f"{len(m)} test images = {len(P)} photos x 4 levels, {len(DISEASES)} diseases; {len(ex)} PlantVillage extras")
print("  photos per disease:", ", ".join(f"{d} {n}" for d, n in P.disease.value_counts().sort_index().items()))
print("  PlantVillage extras:", ", ".join(f"{d} {n}" for d, n in ex.label.value_counts().sort_index().items()))

# The database's noisy copies use the test set's noise recipe (seed family 2026 instead of 299). Check that this
# Colab still rebuilds 40 test images from their clean photo (the research repo: 612/630 identical, rest within 1).
clean_file = dict(zip(P.photo_id, P.image))
sample = pd.concat([g.sample(8, random_state=0) for _, g in m[m.level != "L0"].groupby("noise")])
same = close = 0
for r in sample.itertuples():
    lv = int(r.level[1])
    remade = make_noisy(read_rgb(image_path(cfg, clean_file[r.photo_id])), r.noise, lv, seed_for(r.photo_id, r.noise, lv))
    diff = int(np.abs(remade.astype(int) - read_rgb(image_path(cfg, r.image)).astype(int)).max())
    same, close = same + (diff == 0), close + (diff <= 1)
print(f"Noise recipe check: {same}/{len(sample)} rebuilt test images identical, "
      f"{close}/{len(sample)} within 1 grey level")
if close < 0.9 * len(sample):
    print("WARNING: this Colab does not rebuild the test images (a library changed the noise). Tell Claude.")
NUMBERS["noise_recipe_check"] = {"identical": int(same), "within_1_grey_level": int(close), "of": len(sample)}
done()

## Block A · Train the app model (E4 settings, all 240 photos)

### Step 3 · Train
The same training code as the research repo (`src/safecrop/finetune.py`) with the same settings as E3/E4: the last
4 of CLIP's 24 blocks + the final norm + the projection learn for 15 epochs from clean + noisy views
(cross-entropy + 1.0 × consistency). There are no folds this time: the app learns from **all 240 photos**
(2–3 minutes; the first time, CLIP ViT-L/14 downloads 1.7 GB).

In [ ]:
start("Step 3: train the app model on all 240 photos")
if SMOKE_TEST:                                        # laptop test: a tiny random CLIP saved like a real model
    tiny = WORK / "tiny_clip"
    if not (tiny / "config.json").is_file():
        load_clip(TINY).save_pretrained(tiny)
    cfg["model"], cfg["batch_size"] = str(tiny), 64
    cfg["finetune"].update(epochs=1, num_workers=0)
BASE_MODEL = cfg["model"]                             # openai/clip-vit-large-patch14
ft = cfg["finetune"]
SETTINGS = {"base_model": BASE_MODEL, "trained_blocks": f"last {ft['unfreeze_last']} + final norm + projection",
            "epochs": ft["epochs"], "photos_per_step": ft["batch_photos"], "noisy_views": ft["noisy_views"],
            "training_noise_levels": ft["train_levels"], "flips_and_turns": ft["geometric_aug"], "lr": ft["lr"],
            "head_lr": ft["head_lr"], "weight_decay": ft["weight_decay"], "warmup_epochs": ft["warmup_epochs"],
            "consistency_weight": ft["consistency_weight"], "head_scale": ft["head_scale"], "seed": SEED,
            "noisy_copy_seed": COPY_SEED, "k": K}
for k, v in SETTINGS.items():
    print(f"  {k:24s} {v}")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
LOG = []
enc, head, n_trainable = train_model(cfg, m, P.photo_id.tolist(), DEVICE, SEED, LOG, "app")
pd.DataFrame(LOG).to_csv(OUT / "train_log.csv", index=False)
NUMBERS["app_model"] = {"photos": len(P), "trainable_weights": int(n_trainable),
                        "loss_first_epoch": round(LOG[0]["loss"], 4), "loss_last_epoch": round(LOG[-1]["loss"], 4)}
done(f"{n_trainable / 1e6:.1f} M weights trained; loss {LOG[0]['loss']:.3f} -> {LOG[-1]['loss']:.3f}")

### Step 4 · Save only the changed weights, then load them back like the app
Only the trained parts are saved (~205 MB instead of 1.7 GB). The back end downloads the normal CLIP ViT-L/14 and
puts these on top. To be sure that works, this step does exactly that and checks that photos give the same numbers,
on the GPU and on the CPU (like the app).

In [ ]:
start("Step 4: save the changed weights and load them back like the app")
from safetensors.torch import load_file, save_file
from transformers import CLIPConfig, CLIPVisionModelWithProjection

DELTA = OUT / "clip_ft_delta.safetensors"
RENAME = {"vision.": "vision_model.", "proj.": "visual_projection."}   # our wrapper's names -> CLIP's own names
trained = {n for n, p in enc.named_parameters() if p.requires_grad}
delta = {}
for n, t in enc.state_dict().items():
    if n in trained:
        prefix = next(a for a in RENAME if n.startswith(a))
        delta[RENAME[prefix] + n[len(prefix):]] = t.detach().float().cpu().contiguous()
save_file(delta, str(DELTA), metadata={
    "base_model": BASE_MODEL, "trained": SETTINGS["trained_blocks"], "settings": "E4 defaults, see export_info.json",
    "trained_on": "all 240 photos of the 8 diseases: never score it on the 960 test images",
    "seed": str(SEED), "torch": torch.__version__, "transformers": transformers.__version__, "made": STAMP})
blocks = sorted({int(k.split(".")[3]) for k in delta if ".layers." in k})
print(f"Saved {DELTA.name}: {len(delta)} tensors, {sum(t.numel() for t in delta.values()) / 1e6:.1f} M numbers, "
      f"{DELTA.stat().st_size / 1e6:.0f} MB (blocks {blocks} + post_layernorm + visual_projection)")


def load_app_model(delta_file, device):
    """What the back end does: the normal CLIP image part, then our fine-tuned weights on top."""
    full = CLIPConfig.from_pretrained(BASE_MODEL)
    vcfg = full.vision_config
    vcfg.projection_dim = full.projection_dim         # 768 for ViT-L/14
    model = CLIPVisionModelWithProjection.from_pretrained(BASE_MODEL, config=vcfg)
    missing, unexpected = model.load_state_dict(load_file(str(delta_file)), strict=False)
    assert not unexpected, f"unknown weights in {delta_file}: {unexpected[:3]}"
    return model.float().eval().to(device)


@torch.inference_mode()
def app_embed(model, arrays, batch=64):
    """The app's way to turn photos into numbers: 224 x 224 RGB -> CLIP mean/std -> image part -> projection ->
    unit length, in plain float32 (no fp16), exactly like the back end on a CPU."""
    dev = next(model.parameters()).device
    mean = torch.tensor(CLIP_MEAN, device=dev).view(1, 3, 1, 1)
    std = torch.tensor(CLIP_STD, device=dev).view(1, 3, 1, 1)
    size, out = model.config.image_size, []
    for i in range(0, len(arrays), batch):
        x = torch.from_numpy(np.stack([fit_size(a, size) for a in arrays[i:i + batch]])).to(dev)
        x = (x.permute(0, 3, 1, 2).float() / 255.0 - mean) / std
        v = model.visual_projection(model.vision_model(pixel_values=x).pooler_output)
        out.append(F.normalize(v.float(), dim=-1).cpu())
    return torch.cat(out).numpy() if out else np.zeros((0, model.config.projection_dim), np.float32)


@torch.inference_mode()
def trained_embed(arrays):
    """The trained model itself, in float32 (no fp16)."""
    enc.eval()
    x = torch.from_numpy(np.stack([fit_size(a, enc.size) for a in arrays])).to(DEVICE)
    return enc(x).float().cpu().numpy()


test = [read_rgb(image_path(cfg, im)) for im in P.image.iloc[[0, 120, 239]]]
e_trained = trained_embed(test)
APP = load_app_model(DELTA, DEVICE)                   # also builds the database below
e_app = app_embed(APP, test)
app_cpu = load_app_model(DELTA, torch.device("cpu"))
e_cpu = app_embed(app_cpu, test)
del app_cpu
cos1, dif1 = float((e_trained * e_app).sum(1).min()), float(np.abs(e_trained - e_app).max())
cos2, dif2 = float((e_app * e_cpu).sum(1).min()), float(np.abs(e_app - e_cpu).max())
where = "GPU" if DEVICE.type == "cuda" else "CPU"
print(f"Trained model vs reloaded from the file: cosine {cos1:.6f}, largest difference {dif1:.1e}")
print(f"Reloaded on the {where} vs on the CPU (as in the app): cosine {cos2:.6f}, largest difference {dif2:.1e}")
assert cos1 > 0.99999 and cos2 > 0.9999, "Loading the saved weights changed the numbers: stop and tell Claude."
NUMBERS["reload_check"] = {"cosine_trained_vs_reloaded": round(cos1, 6), "max_difference": dif1,
                           "cosine_gpu_vs_cpu": round(cos2, 6), "max_difference_gpu_vs_cpu": dif2}
REPORT.append(f"### App model\n\nTrained on all {len(P)} photos ({len(DISEASES)} diseases): "
              f"{SETTINGS['trained_blocks']}, {n_trainable / 1e6:.1f} M weights, {ft['epochs']} epochs, seed {SEED}; "
              f"loss {LOG[0]['loss']:.3f} -> {LOG[-1]['loss']:.3f}. The saved weights reload to the same embeddings "
              f"(cosine {cos1:.6f}; {where} vs CPU {cos2:.6f}).\n")
del enc, head
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
save_to_drive("Block A")
done("clip_ft_delta.safetensors saved and checked")

## Block B · Build the picture database

### Step 5 · Make the database pictures
Every photo gets its clean picture + **15 noisy copies** (5 noise types × L1–L3), made with the test-set recipe but
a different seed family (**2026**, never 299): exactly E4's database. The 240 PlantVillage extras get the same:
3 healthy leaves (`healthy_…`), 4 look-alike diseases (`other_…`), and 30 PlantVillage apple scab photos, which stay
**apple scab** (a second photo source, allowed by the research rules).

In [ ]:
start("Step 5: make the 7,680 database pictures")
CROP = {"apple_scab": "apple", "potato_late_blight": "potato", "rice_stem_borer": "rice", "grape_leaf_blight": "grape",
        "pear_leaf_spot": "pear", "jute_mosaic": "jute", "jute_dieback": "jute", "black_gram_yellow_mosaic": "black gram"}
SOURCE = {"apple_scab": ("AppleLeaf9", "CC BY 4.0"), "potato_late_blight": ("PlantVillage", "CC BY-SA 3.0"),
          "rice_stem_borer": ("Mendeley Data hnfjs42d5g", "CC BY 4.0"),
          "black_gram_yellow_mosaic": ("Mendeley Data z55yrbmn2d (IDBGL)", "CC BY 4.0"),
          "pear_leaf_spot": ("DiaMOS Plant", "CC BY 4.0"), "grape_leaf_blight": ("PlantVillage", "CC BY-SA 3.0"),
          "jute_mosaic": ("Harvard Dataverse doi:10.7910/DVN/FJ1DM1", "CC0"),
          "jute_dieback": ("Harvard Dataverse doi:10.7910/DVN/FJ1DM1", "CC0")}
NAME = dict(zip(P.disease, P.disease_name))


def extra_name(option, label):
    if label in NAME:
        return NAME[label]
    if label.startswith("healthy_"):
        return f"Healthy {labels.loc[option, 'crop']} leaf"
    return labels.loc[option, "clip_name"].capitalize()


photos = pd.concat([
    pd.DataFrame({"photo_id": P.photo_id, "label": P.disease, "kind": "disease", "label_name": P.disease_name,
                  "crop": P.disease.map(CROP), "source": [SOURCE[d][0] for d in P.disease],
                  "licence": [SOURCE[d][1] for d in P.disease], "original_file": P.source_file,
                  "in_test_set": True}),
    pd.DataFrame({"photo_id": ex.photo_id, "label": ex.label,
                  "kind": [x.split("_")[0] if x.startswith(("healthy_", "other_")) else "disease" for x in ex.label],
                  "label_name": [extra_name(o, x) for o, x in zip(ex.option, ex.label)],
                  "crop": [labels.loc[o, "crop"] for o in ex.option], "source": "PlantVillage",
                  "licence": "CC BY-SA 3.0", "original_file": [p.split("/")[-1] for p in ex.path],
                  "in_test_set": False}),
], ignore_index=True)
photos["thumb"] = photos.photo_id + ".jpg"

CLEAN = {r.photo_id: read_rgb(image_path(cfg, r.image)) for r in P.itertuples()}       # the 224 x 224 test photos
CLEAN.update({r.photo_id: load_square(r.file, 224) for r in ex.itertuples()})         # 256 px -> central 224 square
assert len(CLEAN) == 480 and all(a.shape == (224, 224, 3) for a in CLEAN.values())

rows = []
for r in photos.itertuples():
    rows.append((r.photo_id, r.label, r.kind, r.in_test_set, "clean", 0, -1))
    for t in NOISE_TYPES:
        for lv in (1, 2, 3):
            rows.append((r.photo_id, r.label, r.kind, r.in_test_set, t, lv, seed_for(r.photo_id, t, lv, COPY_SEED)))
DB = pd.DataFrame(rows, columns=["photo_id", "label", "kind", "in_test_set", "noise", "level", "seed"])
DB["is_copy"] = DB.noise != "clean"

# our copies must be exactly E4's database copies (the same seeds the research code makes)
e4 = noisy_copies_table(m, 1, [1, 2, 3], COPY_SEED)
ours = DB[DB.in_test_set & DB.is_copy]
assert sorted(zip(e4.photo_id, e4.noise, e4.level, e4.seed)) == sorted(zip(ours.photo_id, ours.noise, ours.level, ours.seed))

t0, PICS = time.time(), []
for i, r in enumerate(DB.itertuples()):
    a = CLEAN[r.photo_id]
    PICS.append(a if r.noise == "clean" else make_noisy(a, r.noise, int(r.level), int(r.seed)))
    if (i + 1) % 1920 == 0 or i + 1 == len(DB):
        print(f"  made {i + 1}/{len(DB)} pictures ({time.time() - t0:.0f} s)", flush=True)
print(DB.groupby(["kind", "label"]).agg(photos=("photo_id", "nunique"), pictures=("photo_id", "size")).to_string())
done(f"{len(DB)} pictures = {len(photos)} photos x 16 (our copies match E4's exactly)")

### Step 6 · Turn every picture into 768 numbers and save `database.npz`
With the reloaded app model in plain float32: exactly what the back end computes for a farmer's photo (~4 minutes).

In [ ]:
start("Step 6: embed the database with the app model, save database.npz")


def embed_all(fn, what, chunk=512):
    out, t0 = [], time.time()
    for i in range(0, len(PICS), chunk):
        out.append(fn(PICS[i:i + chunk]))
        if (i // chunk) % 4 == 3 or i + chunk >= len(PICS):
            print(f"  {what}: {min(i + chunk, len(PICS))}/{len(PICS)} pictures ({time.time() - t0:.0f} s)", flush=True)
    return np.concatenate(out)


EMB = embed_all(lambda arrays: app_embed(APP, arrays), "database")
assert np.allclose(np.linalg.norm(EMB, axis=1), 1, atol=1e-4)
DB_FILE = OUT / "database.npz"
DB_INFO = {"base_model": BASE_MODEL, "weights": DELTA.name, "k": K, "noisy_copy_seed": COPY_SEED, "made": STAMP,
           "rows": len(DB), "photos": len(photos), "dims": int(EMB.shape[1]),
           "about": "unit-length float32 embeddings; per photo 1 clean row + 15 noisy copies (5 noise types x L1-L3)"}
np.savez(DB_FILE, emb=EMB.astype(np.float32), label=DB.label.to_numpy(str), photo_id=DB.photo_id.to_numpy(str),
         kind=DB.kind.to_numpy(str), is_copy=DB.is_copy.to_numpy(bool), noise=DB.noise.to_numpy(str),
         level=DB.level.to_numpy(np.int8), seed=DB.seed.to_numpy(np.int64),
         thumb=(DB.photo_id + ".jpg").to_numpy(str), info=np.array(json.dumps(DB_INFO)))

# Sanity check, NOT an accuracy result (this model trained on these photos): with all of its own pictures hidden,
# each clean photo of our 8 diseases should still be voted to its own disease.
q = (DB.in_test_set & ~DB.is_copy).to_numpy()
v = database_vote(EMB[q], DB.photo_id.to_numpy()[q], EMB, DB.label.to_numpy(), DB.photo_id.to_numpy(), K)
sanity = float((v["pred"] == DB.label.to_numpy()[q]).mean())
print(f"Saved {DB_FILE.name}: {EMB.shape[0]:,} pictures x {EMB.shape[1]} numbers, {DB_FILE.stat().st_size / 1e6:.1f} MB")
print(f"Sanity check (not an accuracy result): {sanity:.1%} of the 240 clean photos voted to their own disease")
NUMBERS["database"] = {"rows": len(DB), "photos": len(photos), "dims": int(EMB.shape[1]),
                       "sanity_clean_photos_own_disease": round(sanity, 4)}
done()

### Step 7 · Thumbnails
The app always shows the **clean** photo of a neighbour, even when one of its noisy copies was the match.

In [ ]:
start("Step 7: 200-pixel thumbnails of every clean photo + photos.csv")
THUMB = 200
ex_file = dict(zip(ex.photo_id, ex.file))
for r in photos.itertuples():
    if r.in_test_set:     # our photos exist only as the 224-pixel test pictures
        im = Image.fromarray(CLEAN[r.photo_id]).resize((THUMB, THUMB), Image.LANCZOS)
    else:                 # PlantVillage: straight from the 256-pixel original
        im = Image.fromarray(load_square(ex_file[r.photo_id], THUMB))
    im.save(OUT / "thumbs" / r.thumb, "JPEG", quality=85, optimize=True)
photos.to_csv(OUT / "photos.csv", index=False)
thumbs = sorted((OUT / "thumbs").glob("*.jpg"))
assert len(thumbs) == len(photos)
try:                      # one thumbnail per label, to check the labels by eye
    import matplotlib.pyplot as plt
    first = photos.groupby("label", sort=False).head(1)
    fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
    for ax in axes.flat:
        ax.axis("off")
    for ax, r in zip(axes.flat, first.itertuples()):
        ax.imshow(Image.open(OUT / "thumbs" / r.thumb))
        ax.set_title(r.label.replace("_", " "), fontsize=8)
    plt.tight_layout()
    plt.show() if IN_COLAB else plt.close(fig)
except ImportError:
    pass
save_to_drive("Block B")
done(f"{len(thumbs)} thumbnails, {sum(f.stat().st_size for f in thumbs) / 1e6:.1f} MB")

## Block C · Photo check, cut-offs and the fair checks

### Step 8 · The photo check's code
`photo_check.py` holds the 16 measurements, copied **unchanged** from the research notebook
`SafeCrop_photo_check.ipynb`, plus small helpers for the app. The forests below learn from exactly this code, and the
back end will use the same file.

In [ ]:
start("Step 8: write photo_check.py")
PHOTO_CHECK_PY = r'''"""SafeCrop photo check: what is wrong with a leaf photo (blur, grain, dark, blocky) and how bad (L0-L3).

Made by notebooks/SafeCrop_app_model_export.ipynb (Day 2). The 16 measurements are copied unchanged from the
research notebook SafeCrop_photo_check.ipynb (NabilRafi/SafeCrop, notebooks/previous/), so the two random forests
in photo_check.joblib see exactly the kind of numbers they learned from.

Needs numpy, opencv-python-headless, pillow and joblib, plus the SAME scikit-learn version that trained the
forests (see export_info.json).

    from photo_check import prepare_photo, load_checker, check_photo, needs_resample, fix_resample
    checker = load_checker("photo_check.joblib")
    a = prepare_photo("leaf.jpg")          # central square, 224 x 224, RGB uint8 (CLIP uses the same picture)
    result = check_photo(a, checker)       # {"problem", "problem_words", "problem_sure", "level", "level_sure", "signs"}
"""
import os

import cv2
import numpy as np
from PIL import Image, ImageOps

SIZE = 224
FEATURES = ["brightness", "dark_share", "bright_share", "contrast", "dynamic_range", "saturation", "colourfulness",
            "grain", "grain_relative", "colour_grain", "sharpness", "fine_detail", "edges",
            "blur_direction", "blockiness", "leaf_cover"]
PROBLEM_WORDS = {"clean": "clean", "gaussian_noise": "grain", "low_light": "low light",
                 "jpeg_compression": "compression", "motion_blur": "hand-shake blur", "defocus_blur": "out of focus"}
SIGNS = [   # measurement, which side is bad, what it means (outside the range of 98% of our clean photos)
    ("brightness", "low", "too dark"), ("brightness", "high", "too bright"),
    ("dark_share", "high", "large very dark areas"), ("grain", "high", "grainy (sensor noise)"),
    ("colour_grain", "high", "coloured speckles"), ("sharpness", "low", "blurry"),
    ("fine_detail", "low", "little fine detail"), ("blur_direction", "high", "smeared in one direction (hand shake)"),
    ("blockiness", "high", "blocky (JPEG compression)"), ("saturation", "low", "faded colours"),
    ("dynamic_range", "low", "low contrast (washed out)"), ("leaf_cover", "low", "the leaf fills little of the photo"),
]


def prepare_photo(img, size=SIZE):
    """Any photo (file path, PIL image or array) -> its central square at size x size, RGB uint8.
    The same way the test photos were made: turned by EXIF, central square, bicubic resize."""
    if isinstance(img, (str, os.PathLike)):
        img = Image.open(img)
    if isinstance(img, Image.Image):
        img = ImageOps.exif_transpose(img).convert("RGB")
    else:
        img = Image.fromarray(to_array(img))
    w, h = img.size
    s = min(w, h)
    left, top = (w - s) // 2, (h - s) // 2
    img = img.crop((left, top, left + s, top + s)).resize((size, size), Image.BICUBIC)
    return np.asarray(img, dtype=np.uint8).copy()


# ---------------------------------------------------------------- copied unchanged from the research notebook
def to_array(img):
    """PIL image or array -> uint8 RGB array."""
    if isinstance(img, Image.Image):
        img = np.asarray(img.convert("RGB"))
    a = np.asarray(img)
    if a.ndim == 2:
        a = np.stack([a] * 3, axis=2)
    return np.ascontiguousarray(a[..., :3]).astype(np.uint8)


def luminance(a):
    """brightness of every pixel, 0 (black) to 1 (white)"""
    return (a.astype(np.float32) @ np.array([0.299, 0.587, 0.114], np.float32)) / 255.0


def noise_sigma(g):
    """Immerkaer's fast estimate of random noise (same as the loop notebook's), 0 to 1 scale"""
    k = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], np.float32)
    conv = cv2.filter2D(g.astype(np.float32), -1, k, borderType=cv2.BORDER_REFLECT)[1:-1, 1:-1]
    H, W = g.shape
    return float(np.sqrt(np.pi / 2) * np.abs(conv).sum() / (6 * (W - 2) * (H - 2)))


def fix_resample(a, factor=0.5):
    """shrink then enlarge: each small square of pixels is averaged, which cancels some grain"""
    H, W = a.shape[:2]
    small = cv2.resize(a, (max(8, int(W * factor)), max(8, int(H * factor))), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (W, H), interpolation=cv2.INTER_CUBIC)


def measure(img):
    """16 numbers that describe what is wrong (or right) with a photo"""
    a = to_array(img)
    if max(a.shape[:2]) != 224:
        a = cv2.resize(a, (224, 224), interpolation=cv2.INTER_AREA)
    f = a.astype(np.float32) / 255.0
    g = luminance(a)
    out = {}
    out["brightness"] = float(g.mean())
    out["dark_share"] = float((g < 0.12).mean())
    out["bright_share"] = float((g > 0.95).mean())
    out["contrast"] = float(g.std())
    p2, p98 = np.percentile(g, [2, 98])
    out["dynamic_range"] = float(p98 - p2)
    mx, mn = f.max(2), f.min(2)
    out["saturation"] = float(np.where(mx > 0, (mx - mn) / np.maximum(mx, 1e-6), 0).mean())
    rg, yb = f[..., 0] - f[..., 1], 0.5 * (f[..., 0] + f[..., 1]) - f[..., 2]
    out["colourfulness"] = float(np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean()))
    s = noise_sigma(g)
    out["grain"] = s
    out["grain_relative"] = float(s / max(g.mean(), 0.02))
    ycc = cv2.cvtColor(a, cv2.COLOR_RGB2YCrCb).astype(np.float32) / 255.0
    out["colour_grain"] = float((noise_sigma(ycc[..., 1]) + noise_sigma(ycc[..., 2])) / 2)
    lap = cv2.Laplacian(g, cv2.CV_32F)
    out["sharpness"] = float(np.log(lap.var() + 1e-8))
    P = np.abs(np.fft.fftshift(np.fft.fft2(g - g.mean()))) ** 2
    H, W = g.shape
    yy, xx = np.mgrid[-H // 2:H - H // 2, -W // 2:W - W // 2]
    r = np.hypot(yy / (H / 2), xx / (W / 2))
    band = (r > 0.02) & (r <= 1)
    out["fine_detail"] = float(P[(r > 0.35) & (r <= 1)].sum() / (P[band].sum() + 1e-12))
    u8 = (g * 255).astype(np.uint8)
    med = float(np.median(u8))
    out["edges"] = float((cv2.Canny(u8, int(max(0, 0.66 * med)), int(min(255, 1.33 * med + 1))) > 0).mean())
    # motion blur smears in one direction: compare spectral energy across 12 directions (middle frequencies)
    ang = np.mod(np.arctan2(yy, xx), np.pi)
    mid = (r > 0.08) & (r < 0.5)
    e = np.array([P[mid & (ang >= k * np.pi / 12) & (ang < (k + 1) * np.pi / 12)].mean() for k in range(12)])
    out["blur_direction"] = float(np.log((e.max() + 1e-12) / (e.min() + 1e-12)))
    dh = np.abs(np.diff(g, axis=1))
    dv = np.abs(np.diff(g, axis=0))
    bh = dh[:, 7::8].mean() / (np.delete(dh, np.s_[7::8], axis=1).mean() + 1e-6)
    bv = dv[7::8, :].mean() / (np.delete(dv, np.s_[7::8], axis=0).mean() + 1e-6)
    out["blockiness"] = float((bh + bv) / 2)
    exg = 2 * f[..., 1] - f[..., 0] - f[..., 2]
    out["leaf_cover"] = float((exg > 0.05).mean())
    return out


# ---------------------------------------------------------------- the check, for the app
def load_checker(path):
    """The forests + normal ranges saved by the notebook (needs the same scikit-learn version)."""
    import joblib
    return joblib.load(path)


def signs_of(m, ranges):
    """Plain words for every measurement outside the range of 98% of our clean photos."""
    out = []
    for k, side, words in SIGNS:
        lo, hi = ranges[k]["low"], ranges[k]["high"]
        if (side == "low" and m[k] < lo) or (side == "high" and m[k] > hi):
            out.append(words)
    return out


def check_photo(a, checker):
    """a: 224 x 224 RGB uint8 from prepare_photo. The main problem and its level, how sure, and plain-word signs."""
    m = measure(a)
    x = np.array([[m[k] for k in checker["features"]]], dtype=np.float64)
    pp = checker["problem"].predict_proba(x)[0]
    lp = checker["level"].predict_proba(x)[0]
    i, j = int(pp.argmax()), int(lp.argmax())
    problem = str(checker["problem"].classes_[i])
    return {"problem": problem, "problem_words": PROBLEM_WORDS.get(problem, problem), "problem_sure": float(pp[i]),
            "level": str(checker["level"].classes_[j]), "level_sure": float(lp[j]),
            "signs": signs_of(m, checker["ranges"]), "measurements": m}


def needs_resample(result, sure_cut=0.6):
    """The research rule: resample only blocky (JPEG-compressed) photos. When the forest is unsure, the plain-word
    signs decide. (The app uses it only if thresholds.json says use_resample.)"""
    if result["problem_sure"] >= sure_cut:
        return result["problem"] == "jpeg_compression"
    return "blocky (JPEG compression)" in result["signs"]
'''
(OUT / "photo_check.py").write_text(PHOTO_CHECK_PY, encoding="utf-8")
sys.dont_write_bytecode = True                        # no __pycache__ folder inside the output folder
spec = importlib.util.spec_from_file_location("photo_check", OUT / "photo_check.py")
pc = importlib.util.module_from_spec(spec)
sys.modules["photo_check"] = pc
spec.loader.exec_module(pc)
done(f"photo_check.py written ({len(pc.FEATURES)} measurements)")

### Step 9 · Train the photo check on all 960 images
Two random forests (500 trees each): one names the problem (clean, grain, hand-shake blur, out of focus, low light,
compression), one the level (L0–L3). First a fair test (5 folds by photo: every image is judged by forests that never
saw its photo), then the final forests learn from all 960 images. Normal range = where 98% of the 240 clean photos fall.

In [ ]:
start("Step 9: train the photo check on all 960 images")
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict

t0 = time.time()
FM = pd.DataFrame([pc.measure(read_rgb(image_path(cfg, im))) for im in m.image])[pc.FEATURES]
print(f"  measured {len(FM)} images ({time.time() - t0:.0f} s)")
saved = pd.read_csv(RESEARCH / "data" / "baselines" / "photo_check_quality_measurements.csv").set_index("image")
both = m.image.isin(saved.index).to_numpy()
close_rows = np.isclose(FM[both].to_numpy(), saved.loc[m.image[both], pc.FEATURES].to_numpy(),
                        rtol=1e-3, atol=1e-6).all(1)
print(f"  the same numbers as the research notebook for {int(close_rows.sum())}/{int(both.sum())} images")

X = FM.to_numpy(np.float64)
y_problem = np.where(m.level == "L0", "clean", m.noise).astype(str)
y_level = m.level.to_numpy(str)
groups = m.photo_id.to_numpy()


def forest():
    return RandomForestClassifier(n_estimators=500, random_state=0, n_jobs=-1)


folds = GroupKFold(n_splits=5)
oof_p = cross_val_predict(forest(), X, y_problem, cv=folds, groups=groups)
oof_l = cross_val_predict(forest(), X, y_level, cv=folds, groups=groups)
acc_p, acc_l = float((oof_p == y_problem).mean()), float((oof_l == y_level).mean())
within1 = float((np.abs(np.array([int(s[1]) for s in oof_l]) - np.array([int(s[1]) for s in y_level])) <= 1).mean())
by_level = {lv: float((oof_p == y_problem)[y_level == lv].mean()) for lv in ["L0", "L1", "L2", "L3"]}
conf = pd.crosstab(pd.Series(y_problem).map(pc.PROBLEM_WORDS).rename("real problem"),
                   pd.Series(oof_p).map(pc.PROBLEM_WORDS).rename("found"))
print(f"Fair test (5 folds by photo): problem right {acc_p:.1%}, level right {acc_l:.1%} "
      f"(within one level {within1:.1%})")
print("  problem right by level:", ", ".join(f"{k} {v:.1%}" for k, v in by_level.items()))
print(conf.to_string())

fp, fl = forest().fit(X, y_problem), forest().fit(X, y_level)
fp.n_jobs = fl.n_jobs = None                          # one thread when the app checks one photo
qs = FM[m.level.to_numpy() == "L0"].quantile([0.01, 0.5, 0.99])
RANGES = {k: {"low": float(qs.loc[0.01, k]), "typical": float(qs.loc[0.5, k]), "high": float(qs.loc[0.99, k])}
          for k in pc.FEATURES}
CHECKER = {"problem": fp, "level": fl, "features": list(pc.FEATURES), "ranges": RANGES, "sure_cut": PHOTO_CHECK_SURE,
           "sklearn_version": sklearn.__version__, "numpy_version": np.__version__, "made": STAMP,
           "trained_on": "960 images = 240 photos x L0-L3 (8 diseases)",
           "fair_test": {"problem": round(acc_p, 4), "level": round(acc_l, 4), "level_within_one": round(within1, 4)}}
joblib.dump(CHECKER, OUT / "photo_check.joblib", compress=3)

chk = pc.load_checker(OUT / "photo_check.joblib")     # the saved file, used the way the app will
print("\nThe saved photo check on one clean photo and one L3 photo of each problem:")
for r in [m[m.level == "L0"].iloc[0]] + [m[(m.noise == t) & (m.level == "L3")].iloc[0] for t in NOISE_TYPES]:
    res = pc.check_photo(pc.prepare_photo(image_path(cfg, r.image)), chk)
    print(f"  {Path(r.image).name:48s} -> {res['problem_words']} ({res['problem_sure']:.0%} sure), level "
          f"{res['level']} ({res['level_sure']:.0%}); resample: {'yes' if pc.needs_resample(res, PHOTO_CHECK_SURE) else 'no'}")
PINS = (f"scikit-learn=={sklearn.__version__}  numpy=={np.__version__}  "
        f"opencv-python-headless=={cv2.__version__.split('-')[0]}.*")
print(f"\nPIN THESE in backend/requirements.txt: {PINS}")
REPORT.append("### Photo check (16 measurements + 2 random forests of 500 trees)\n\n"
              f"Fair test on 960 images (5 folds by photo): problem right **{acc_p:.1%}**, level right "
              f"**{acc_l:.1%}** (within one level {within1:.1%}). Problem right by level: "
              + ", ".join(f"{k} {v:.1%}" for k, v in by_level.items())
              + f". Measurements equal to the research notebook's for {int(close_rows.sum())}/{int(both.sum())} images. "
              f"scikit-learn {sklearn.__version__}.\n\n" + md_table(conf.reset_index()) + "\n")
NUMBERS["photo_check"] = {**CHECKER["fair_test"], "problem_by_level": {k: round(v, 4) for k, v in by_level.items()},
                          "same_as_research": f"{int(close_rows.sum())}/{int(both.sum())}"}
done()

### Step 10 · The "sure" and "not sure" cut-offs
From the research repo's **out-of-fold E4 answers** (5 folds: every answer comes from a model that never saw that
photo), *not* from the new model. An always-rising curve of "how often right" is fitted against the vote share
(isotonic regression). **Sure** starts where the curve reaches 95%; **not sure** is everything below the point where
it reaches 70%. By default the recipe-B answers decide (noise made a different way, so closer to real photos).

In [ ]:
start("Step 10: the 'sure' and 'not sure' cut-offs")
from sklearn.isotonic import IsotonicRegression

truth = pd.concat([pd.read_csv(RESEARCH / "data" / f) for f in
                   ["manifest.csv", "manifest_black_gram.csv", "manifest_recipe_b.csv"]]).drop_duplicates("image")
truth = truth.set_index("image")


def oof_answers(which):
    p = pd.read_csv(RESEARCH / BUNDLE["e4_out_of_fold"][which])
    p["correct"] = p.pred_label.to_numpy() == truth.loc[p.image, "disease"].to_numpy()
    p["level"] = truth.loc[p.image, "level"].to_numpy()
    p["vote"] = p.confidence.astype(float)
    return p


OOF = {"test": oof_answers("test"), "recipe_b": oof_answers("recipe_b")}
OOF["both"] = pd.concat([OOF["test"], OOF["recipe_b"]], ignore_index=True)


def cut_for(p, target):
    """The vote share where the always-rising 'how often right' curve first reaches `target`
    (half-way between the two nearest vote shares seen)."""
    iso = IsotonicRegression(increasing=True, y_min=0, y_max=1, out_of_bounds="clip")
    iso.fit(p.vote, p.correct.astype(float))
    xs = np.unique(p.vote)
    hit = np.flatnonzero(iso.predict(xs) >= target)
    if len(hit) == 0:
        return 1.01                                   # never that accurate: nothing qualifies
    i = hit[0]
    return float(xs[0]) if i == 0 else float((xs[i - 1] + xs[i]) / 2)


CUTS = {w: (round(cut_for(p, SURE_ACCURACY), 5), round(cut_for(p, NOT_SURE_ACCURACY), 5)) for w, p in OOF.items()}
SURE, NOT_SURE = CUTS[THRESHOLDS_FROM]
assert NOT_SURE <= SURE

rows = []
for name, lo, hi in [("all 5 agree", 0.9, 1.01), ("about 4 of 5", 0.7, 0.9), ("about 3 of 5", 0.5, 0.7),
                     ("2 or fewer", 0.0, 0.5)]:
    row = {"similar pictures that agree": name, "vote share": f"{lo:.1f} to {min(hi, 1.0):.1f}"}
    for w in ["test", "recipe_b"]:
        p = OOF[w]
        row[f"right ({w})"] = right(p.correct, (p.vote >= lo) & (p.vote < hi))
    rows.append(row)
band_table = pd.DataFrame(rows)
rows = []
for w, p in OOF.items():
    ok, v = p.correct.to_numpy(), p.vote.to_numpy()
    rows.append({"answers": w, "sure": right(ok, v >= SURE), "likely": right(ok, (v >= NOT_SURE) & (v < SURE)),
                 "not sure": right(ok, v < NOT_SURE)})
dec_table = pd.DataFrame(rows)
print(f"Chosen from the {THRESHOLDS_FROM} answers ({len(OOF[THRESHOLDS_FROM])}): sure = vote share >= {SURE}, "
      f"not sure = vote share < {NOT_SURE}")
print("  (the other answer sets would give: " + "; ".join(f"{w}: sure {s}, not sure {n}" for w, (s, n) in CUTS.items()) + ")")
print(band_table.to_string(index=False))
print("\nHow the three decisions would have split the out-of-fold answers:")
print(dec_table.to_string(index=False))
REPORT.append(f"### Cut-offs: sure / likely / not sure\n\nFrom the research repo's out-of-fold E4 answers "
              f"(8 diseases, 5 folds), isotonic fit on **{THRESHOLDS_FROM}**: **sure** = vote share >= {SURE}, "
              f"**not sure** = vote share < {NOT_SURE}, likely in between. (Other answer sets would give: "
              + "; ".join(f"{w}: {s} / {n}" for w, (s, n) in CUTS.items()) + ".)\n\n" + md_table(band_table)
              + "\n\n" + md_table(dec_table) + "\n")
NUMBERS["cut_offs"] = {"from": THRESHOLDS_FROM, "sure": SURE, "not_sure": NOT_SURE,
                       "all_sets": {w: {"sure": s, "not_sure": n} for w, (s, n) in CUTS.items()}}
done(f"sure >= {SURE}, not sure < {NOT_SURE}")

### The fair checks (fold 0)
The app model has seen every photo, so it cannot be tested on them. The next steps use a second model trained exactly
like the research repo's E4 fold 0: on 192 photos, holding out the **48 photos of fold 0** (42 + 6 black gram), same
seed. It answers 192 held-out test images (48 photos × L0–L3). About 8 minutes in total.

### Step 11 · Train the fold-0 model and build its database

In [ ]:
start("Step 11: the fold-0 model and its database")
train_ids, test_ids = fold_photos(m, 0)
assert len(train_ids) == 192 and len(test_ids) == 48
LOG0 = []
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
enc0, head0, _ = train_model(cfg, m, train_ids, DEVICE, SEED + 0, LOG0, "0")   # the repo used seed 299 + fold
del head0
pd.DataFrame(LOG + LOG0).to_csv(OUT / "train_log.csv", index=False)
emb0 = lambda arrays: embed_arrays(enc0, arrays, DEVICE, cfg["batch_size"])    # the research repo's way, as in E4
R0 = embed_all(emb0, "fold-0 database")
Q = m[m.photo_id.isin(test_ids)].reset_index(drop=True)                        # the 192 held-out test images
Q0 = emb0([read_rgb(image_path(cfg, im)) for im in Q.image])
r_label, r_photo = DB.label.to_numpy(), DB.photo_id.to_numpy()
truth0, lev0 = Q.disease.to_numpy(), Q.level.to_numpy()
done(f"held out {len(test_ids)} photos = {len(Q)} test images")

### Step 12 · Do the healthy / look-alike pictures hurt?
The same 192 held-out images, answered with three databases. For comparison: in the frozen noisy album, adding them
took L3 from 95.2% to 88.1%.

In [ ]:
start("Step 12: do the healthy / look-alike pictures hurt? (fold 0)")
APP_DB = "+ all 240 PlantVillage extras (= the app)"
VARIANTS = {"our 8 diseases only (E4 as in the repo)": DB.in_test_set.to_numpy(),
            "+ 210 healthy / look-alike pictures": (DB.in_test_set | DB.kind.isin(["healthy", "other"])).to_numpy(),
            APP_DB: np.ones(len(DB), bool)}
VOTES = {name: database_vote(Q0, Q.photo_id.to_numpy(), R0[sel], r_label[sel], r_photo[sel], K)
         for name, sel in VARIANTS.items()}
old7 = Q.disease.to_numpy() != "black_gram_yellow_mosaic"
s3 = lev0 == "L3"
rows = []
for name, v in VOTES.items():
    ok = v["pred"] == truth0
    rows.append({"database": name, **{lv: right(ok, lev0 == lv) for lv in ["L0", "L1", "L2", "L3"]},
                 "L3, old 7 diseases": right(ok, s3 & old7),
                 "L3 answered healthy/other": int(pd.Series(v["pred"][s3]).str.startswith(("healthy_", "other_")).sum())})
repo = OOF["test"][OOF["test"].fold == 0]            # the research repo's own E4 run on the same 48 photos
rok, rlev = repo.correct.to_numpy(), repo.level.to_numpy()
rows.append({"database": "research repo's E4 run, fold 0 (for comparison)",
             **{lv: right(rok, rlev == lv) for lv in ["L0", "L1", "L2", "L3"]},
             "L3, old 7 diseases": right(rok, (rlev == "L3") & ~repo.image.str.contains("black_gram").to_numpy()),
             "L3 answered healthy/other": "-"})
healthy_table = pd.DataFrame(rows)
a_ok = VOTES[APP_DB]["pred"] == truth0
b_ok = VOTES["our 8 diseases only (E4 as in the repo)"]["pred"] == truth0
only_app, only_ours = int((a_ok & ~b_ok)[s3].sum()), int((~a_ok & b_ok)[s3].sum())
p_l3 = binomtest(only_app, only_app + only_ours, 0.5).pvalue if only_app + only_ours else 1.0
print(healthy_table.to_string(index=False))
print(f"\nL3, the app's database vs ours only: only the app's right {only_app}, only ours right {only_ours}, "
      f"McNemar p = {p_l3:.3f}")

share = VOTES[APP_DB]["confidence"]
dec = np.where(share >= SURE, "sure", np.where(share >= NOT_SURE, "likely", "not sure"))
dec0_table = pd.DataFrame([{"decision": d, "all levels": right(a_ok, dec == d), "L3": right(a_ok, (dec == d) & s3)}
                           for d in ["sure", "likely", "not sure"]])
print("\nWith the chosen cut-offs, the fold-0 answers (the app's database) split like this:")
print(dec0_table.to_string(index=False))
per = Q[["image", "photo_id", "disease", "level", "noise"]].copy()
for short, name in zip(["ours", "plus210", "app"], VOTES):
    per[f"pred_{short}"], per[f"vote_{short}"] = VOTES[name]["pred"], np.round(VOTES[name]["confidence"], 4)
per.to_csv(OUT / "checks" / "fold0_answers.csv", index=False)
REPORT.append("### Fold 0: do the healthy / look-alike pictures hurt? (48 held-out photos, 192 images)\n\n"
              + md_table(healthy_table)
              + f"\n\nL3, the app's database vs ours only: only the app's right {only_app}, only ours right "
              f"{only_ours} (McNemar p = {p_l3:.3f}). Frozen noisy album for comparison: 95.2% -> 88.1% at L3.\n\n"
              "The chosen cut-offs on these fold-0 answers (the app's database):\n\n" + md_table(dec0_table) + "\n")
NUMBERS["fold0_healthy_check"] = {"table": rows, "L3_only_app_right": only_app, "L3_only_ours_right": only_ours,
                                  "L3_mcnemar_p": round(float(p_l3), 4)}
done()

### Step 13 · Does the resample fix help compressed photos?
Resample = shrink to half size and enlarge again: the only photo edit that helped the frozen model (57% → 79% on
compressed L3 photos). The app keeps it only if it **helps or ties** for E4 on the compressed L3 photos of fold 0.

In [ ]:
start("Step 13: does resample help compressed photos? (fold 0)")
jp = Q[(Q.noise == "jpeg_compression") & (Q.level != "L0")]
Qr = emb0([pc.fix_resample(read_rgb(image_path(cfg, im))) for im in jp.image])
sel = VARIANTS[APP_DB]
with_fix = database_vote(Qr, jp.photo_id.to_numpy(), R0[sel], r_label[sel], r_photo[sel], K)["pred"] == jp.disease.to_numpy()
without = a_ok[jp.index.to_numpy()]
rows = []
for lv in ["L1", "L2", "L3"]:
    s = (jp.level == lv).to_numpy()
    rows.append({"level": lv, "compressed images": int(s.sum()), "right without resample": int(without[s].sum()),
                 "right with resample": int(with_fix[s].sum()), "fixed": int((with_fix & ~without)[s].sum()),
                 "broken": int((~with_fix & without)[s].sum())})
resample_table = pd.DataFrame(rows)
l3 = {k: int(v) for k, v in resample_table.set_index("level").loc["L3"].items()}
USE_RESAMPLE = l3["right with resample"] >= l3["right without resample"]
print(resample_table.to_string(index=False))
print(f"\nDecision: {'KEEP' if USE_RESAMPLE else 'DROP'} the resample fix in the app (L3: {l3['right with resample']} "
      f"right with vs {l3['right without resample']} without, of {l3['compressed images']}; rule: keep if it helps or ties)")
if l3["compressed images"] < 20:
    print(f"  Note: only {l3['compressed images']} compressed L3 images in fold 0: too few to show a real difference.")
REPORT.append("### Fold 0: does resample help compressed photos? (the app's database)\n\n" + md_table(resample_table)
              + f"\n\nDecision: **{'keep' if USE_RESAMPLE else 'drop'}** resample (rule: keep if it helps or ties at L3).\n")
NUMBERS["fold0_resample_check"] = {"table": rows, "use_resample": USE_RESAMPLE}
done()

### Step 14 · The "not a leaf" cut-off
For every photo the app looks at its best similarity to any database picture. Leaves (the 192 held-out images, never
seen by the fold-0 model) are compared with pictures that are not leaves (yours from `non_leaf/` + built-in sample
pictures). The cut-off is the value that sorts the most of both groups correctly (each group counts equally).
Then the app model is checked with the same cut-off.

In [ ]:
start("Step 14: the 'not a leaf' cut-off")
NONLEAF = []                                           # (name, from where, 224 x 224 photo)
folder = BASE / NON_LEAF_DIR
if folder.is_dir():
    for f in sorted(folder.iterdir()):
        if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".heic", ".heif", ".bmp"}:
            try:
                NONLEAF.append((f.name, "your photos", pc.prepare_photo(f)))
            except Exception as e:
                print(f"  skipped {f.name}: {e}")
    print(f"  {len(NONLEAF)} of your photos from {folder}")
else:
    print(f"  no folder {folder} (that's OK: only built-in sample pictures are used)")
try:
    import skimage.data as skdata
    for name in ["astronaut", "camera", "chelsea", "coffee", "coins", "rocket", "hubble_deep_field", "moon",
                 "page", "text", "brick", "gravel", "clock", "colorwheel"]:
        try:
            a = np.asarray(getattr(skdata, name)())
            if a.dtype != np.uint8:
                a = (a * 255 if a.max() <= 1 else a).astype(np.uint8)
            NONLEAF.append((name, "built-in sample", pc.prepare_photo(a)))
        except Exception as e:
            print(f"  (sample picture {name} not available: {type(e).__name__})")
except ImportError:
    print("  scikit-image is missing: no built-in sample pictures")
kinds = np.array([k for _, k, _ in NONLEAF])
non_arrays = [a for _, _, a in NONLEAF]

best_non0 = (emb0(non_arrays) @ R0.T).max(1) if NONLEAF else np.zeros(0)
S = Q0 @ R0.T
S[Q.photo_id.to_numpy(str)[:, None] == r_photo.astype(str)[None, :]] = -np.inf          # never its own photo
best_leaf0 = S.max(1)


def pick_cut(leaf, nonleaf):
    """The similarity that sorts the most pictures correctly, leaves and non-leaves counting equally
    (if a range of values ties, its middle)."""
    vals = np.unique(np.concatenate([leaf, nonleaf]))
    cands = (vals[:-1] + vals[1:]) / 2
    score = np.array([(np.mean(leaf >= c) + np.mean(nonleaf < c)) / 2 for c in cands])
    best = np.flatnonzero(score >= score.max() - 1e-12)
    return float(cands[best[len(best) // 2]]), float(score.max())


if len(best_non0):
    NOT_A_LEAF, balanced = pick_cut(best_leaf0, best_non0)
    how = (f"fold-0 model: {len(best_leaf0)} held-out leaf images vs {len(best_non0)} non-leaf pictures "
           f"(balanced accuracy {balanced:.3f})")
else:
    NOT_A_LEAF = float(np.quantile(best_leaf0, 0.01))
    how = "NO non-leaf pictures: fallback that keeps 99% of the held-out leaf images"
NOT_A_LEAF = round(NOT_A_LEAF, 5)
per["best_similarity"] = np.round(best_leaf0, 4)
per.to_csv(OUT / "checks" / "fold0_answers.csv", index=False)

rows = []
for lv in ["L0", "L1", "L2", "L3"]:
    s = lev0 == lv
    rows.append({"pictures": f"leaves {lv} (held out)", "called 'not a leaf'": right(best_leaf0 < NOT_A_LEAF, s),
                 "best similarity": f"{best_leaf0[s].min():.3f} to {best_leaf0[s].max():.3f}"})
for k in ["your photos", "built-in sample"]:
    s = kinds == k
    if s.any():
        rows.append({"pictures": f"not leaves: {k}", "called 'not a leaf'": right(best_non0 < NOT_A_LEAF, s),
                     "best similarity": f"{best_non0[s].min():.3f} to {best_non0[s].max():.3f}"})
leaf_table = pd.DataFrame(rows)
print(f"Cut-off: best similarity below {NOT_A_LEAF} = probably not a leaf ({how})")
print(leaf_table.to_string(index=False))
if NONLEAF:
    top = np.argsort(-best_non0)[:5]
    print("  most leaf-like non-leaf pictures:", ", ".join(f"{NONLEAF[i][0]} {best_non0[i]:.3f}" for i in top))

# the app model with the same cut-off (its database file, loaded as the app will)
del enc0, R0, S
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
D = np.load(OUT / "database.npz")
E, D_photo = D["emb"], D["photo_id"].astype(str)
best_non_app = (app_embed(APP, non_arrays) @ E.T).max(1) if NONLEAF else np.zeros(0)
SA = app_embed(APP, [read_rgb(image_path(cfg, im)) for im in m.image]) @ E.T
SA[m.photo_id.to_numpy(str)[:, None] == D_photo[None, :]] = -np.inf
best_leaf_app = SA.max(1)
lev_all = m.level.to_numpy()
rows = [{"pictures": f"test images {lv} (seen in training: optimistic)",
         "called 'not a leaf'": right(best_leaf_app < NOT_A_LEAF, lev_all == lv)} for lv in ["L0", "L1", "L2", "L3"]]
for k in ["your photos", "built-in sample"]:
    s = kinds == k
    if s.any():
        rows.append({"pictures": f"not leaves: {k}", "called 'not a leaf'": right(best_non_app < NOT_A_LEAF, s)})
app_leaf_table = pd.DataFrame(rows)
print("\nThe app model with the same cut-off:")
print(app_leaf_table.to_string(index=False))
if NONLEAF and np.mean(best_non_app >= NOT_A_LEAF) > 0.2:
    print("WARNING: the app model lets more than 20% of the non-leaf pictures through. Tell Claude.")
pd.DataFrame({"name": [n for n, _, _ in NONLEAF], "from": kinds, "best_similarity_fold0": np.round(best_non0, 4),
              "best_similarity_app": np.round(best_non_app, 4)}).to_csv(OUT / "checks" / "not_a_leaf.csv", index=False)
REPORT.append(f"### The 'not a leaf' cut-off\n\nBest similarity below **{NOT_A_LEAF}** = probably not a leaf ({how}).\n\n"
              + md_table(leaf_table) + "\n\nThe app model with the same cut-off:\n\n" + md_table(app_leaf_table) + "\n")
NUMBERS["not_a_leaf"] = {"cut": NOT_A_LEAF, "how": how, "non_leaf_pictures": len(NONLEAF),
                         "your_photos": int((kinds == "your photos").sum())}
done(f"not a leaf: best similarity < {NOT_A_LEAF}")

### Step 15 · Save `thresholds.json`, `README.md` and `export_info.json`

In [ ]:
start("Step 15: save thresholds.json, README.md and export_info.json")
import hashlib

THRESHOLDS = {
    "k": K, "sure": SURE, "not_sure": NOT_SURE, "not_a_leaf": NOT_A_LEAF, "photo_check_sure": PHOTO_CHECK_SURE,
    "use_resample": bool(USE_RESAMPLE),
    "explain": {
        "sure": f"vote share >= sure: 'Very likely' (out-of-fold E4 answers there were right at least "
                f"{SURE_ACCURACY:.0%} of the time)",
        "not_sure": f"vote share < not_sure: 'Not sure' (right less than {NOT_SURE_ACCURACY:.0%}); in between: 'Likely'",
        "not_a_leaf": "best similarity to any database picture below this: probably not a leaf, ask for a retake",
        "photo_check_sure": "trust the photo check's main problem only at or above this probability",
        "use_resample": "shrink + enlarge blocky (JPEG-compressed) photos before CLIP (photo_check.needs_resample)",
        "vote_share": "the share of the similarity-weighted vote of the k most similar pictures won by the top label"},
    "chosen_from": {
        "sure_and_not_sure": f"isotonic fit on the research repo's out-of-fold E4 answers ({THRESHOLDS_FROM}, "
                             f"{len(OOF[THRESHOLDS_FROM])} answers, 8 diseases, 5 folds)",
        "not_a_leaf": how,
        "use_resample": f"fold 0, compressed L3 images: {l3['right with resample']} right with resample vs "
                        f"{l3['right without resample']} without (keep if it helps or ties)"},
    "made": STAMP}
(OUT / "thresholds.json").write_text(json.dumps(THRESHOLDS, indent=2), encoding="utf-8")


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def size_of(name):
    p = OUT / name
    b = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) if p.is_dir() else p.stat().st_size
    return f"{b / 1e6:.1f} MB" if b >= 1e5 else f"{b / 1e3:.0f} KB"


shutil.rmtree(OUT / "__pycache__", ignore_errors=True)   # never ship Python's cache files


FILES = [("clip_ft_delta.safetensors", "fine-tuned last 4 blocks + final norm + projection of CLIP ViT-L/14 (goes on "
                                       "top of openai/clip-vit-large-patch14)"),
         ("database.npz", f"{len(DB):,} pictures x {EMB.shape[1]} numbers + label, photo_id, kind, is_copy, noise, "
                          "level, seed, thumb, info"),
         ("thumbs", f"{len(photos)} clean photos, 200 x 200 JPEG, named <photo_id>.jpg"),
         ("photos.csv", "one row per photo: label, name, crop, source, licence, original file, thumbnail"),
         ("photo_check.joblib", f"two random forests + normal ranges (scikit-learn {sklearn.__version__}: pin it)"),
         ("photo_check.py", "the 16 measurements + prepare_photo, check_photo, needs_resample, fix_resample"),
         ("thresholds.json", "the cut-offs: sure, not sure, not a leaf, photo-check trust, resample yes/no"),
         ("train_log.csv", "loss per epoch (app model and fold-0 model)"),
         ("checks", "per-image answers of the fold-0 checks and the not-a-leaf check")]
LOAD_SNIPPET = '''import json, numpy as np, torch, torch.nn.functional as F
from safetensors.torch import load_file
from transformers import CLIPConfig, CLIPVisionModelWithProjection
from photo_check import prepare_photo, load_checker, check_photo, needs_resample, fix_resample

t = json.load(open("thresholds.json"))
BASE = "openai/clip-vit-large-patch14"
cfg = CLIPConfig.from_pretrained(BASE)
vcfg = cfg.vision_config
vcfg.projection_dim = cfg.projection_dim                 # 768
model = CLIPVisionModelWithProjection.from_pretrained(BASE, config=vcfg).float().eval()
missing, unexpected = model.load_state_dict(load_file("clip_ft_delta.safetensors"), strict=False)
assert not unexpected                                    # ours replace the last 4 blocks + final norm + projection
db = np.load("database.npz")                             # emb (N x 768, unit length), label, photo_id, thumb, ...
MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(1, 3, 1, 1)
STD = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(1, 3, 1, 1)

a = prepare_photo("leaf.jpg")                            # EXIF turn, central square, 224 x 224 RGB uint8
check = check_photo(a, load_checker("photo_check.joblib"))
if t["use_resample"] and needs_resample(check, t["photo_check_sure"]):
    a = fix_resample(a)
x = (torch.from_numpy(a).permute(2, 0, 1)[None].float() / 255 - MEAN) / STD
with torch.inference_mode():
    q = F.normalize(model.visual_projection(model.vision_model(pixel_values=x).pooler_output), dim=-1)[0].numpy()
sims = db["emb"] @ q                                     # cosine similarity with every database picture
top5 = np.argsort(-sims)[:5]                             # weighted vote: similarity summed per label -> vote share'''
c7 = BUNDLE["research_commit"][:7]
lines = [
    "# SafeCrop app model files", "",
    f"Made {STAMP} by `notebooks/SafeCrop_app_model_export.ipynb` (repo safecrop-app) from NabilRafi/SafeCrop commit "
    f"`{c7}`, on {GPU}." + ("  **SMOKE TEST with a tiny fake model: do not use these files.**" if SMOKE_TEST else ""), "",
    "**Never score this model on the 960 test images: it trained on every one of their photos.** Accuracy for "
    "reports = the 5-fold results of the research repo. The checks below use a separate fold-0 model that never saw "
    "48 of the photos.", "",
    "## Files", "", "| File | What it is | Size |", "|---|---|---|",
    *[f"| `{n}` | {d} | {size_of(n)} |" for n, d in FILES], "",
    "## Cut-offs (`thresholds.json`)", "", "| Cut-off | Value | Meaning |", "|---|---|---|",
    *[f"| `{k}` | {THRESHOLDS[k]} | {THRESHOLDS['explain'][k]} |"
      for k in ["sure", "not_sure", "not_a_leaf", "photo_check_sure", "use_resample"]],
    f"| `k` | {K} | the {K} most similar database pictures vote, weighted by similarity; vote share = "
    "the winning label's share of that vote |", "",
    "## How the back end uses these files", "", "```python", LOAD_SNIPPET, "```", "",
    "## Numbers (for the model card and the slides)", "", *REPORT,
    "## Settings", "", "| Setting | Value |", "|---|---|", *[f"| {k} | {v} |" for k, v in SETTINGS.items()], "",
    "## Library versions", "", "| Library | Version |", "|---|---|", *[f"| {k} | {v} |" for k, v in VERSIONS.items()], "",
    f"Pin these in `backend/requirements.txt`: `{PINS}`. The photo-check forests only load with the same "
    "scikit-learn (and numpy major) version, and the 16 measurements can shift slightly between OpenCV versions.", ""]
(OUT / "README.md").write_text("\n".join(lines), encoding="utf-8")
INFO = {"made": STAMP, "smoke_test": SMOKE_TEST, "notebook": "safecrop-app/notebooks/SafeCrop_app_model_export.ipynb",
        "research": {"repo": BUNDLE["research_repo"], "commit": BUNDLE["research_commit"], "inputs_zip": ZIP.name,
                     "inputs_made": BUNDLE["made"]},
        "versions": VERSIONS, "settings": SETTINGS, "thresholds": {k: v for k, v in THRESHOLDS.items() if k != "explain"},
        "numbers": NUMBERS, "seconds_per_step": TIMES,
        "files": {p.relative_to(OUT).as_posix(): {"bytes": p.stat().st_size, "sha256": sha256(p)}
                  for p in sorted(OUT.rglob("*")) if p.is_file() and p.parent.name != "thumbs"},
        "thumbs": {"files": len(thumbs), "bytes": sum(f.stat().st_size for f in thumbs)}}
(OUT / "export_info.json").write_text(json.dumps(INFO, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o)),
                                      encoding="utf-8")
print(json.dumps({k: v for k, v in THRESHOLDS.items() if k not in ("explain", "chosen_from")}, indent=1))
done("README.md is the model card draft: every number is in it")

### Step 16 · Zip, Google Drive and Hugging Face

In [ ]:
start("Step 16: zip, Google Drive, Hugging Face")
save_to_drive("all files")
zip_path = Path(shutil.make_archive(str(WORK / OUT_NAME), "zip", root_dir=OUT))
print(f"Zip: {zip_path} ({zip_path.stat().st_size / 1e6:.0f} MB)")
if DRIVE_OUT is not None:
    target = BASE / zip_path.name
    if target.exists():
        target.rename(target.with_name(f"{OUT_NAME}_old_{datetime.datetime.now(DHAKA):%Y%m%d-%H%M%S}.zip"))
    shutil.copyfile(zip_path, target)
    print("  copied to Google Drive:", target)
HF_URL = ""
if UPLOAD_TO_HF and not SMOKE_TEST:
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception as e:
        print(f"Hugging Face upload skipped: no Colab secret HF_TOKEN with notebook access ({type(e).__name__}).")
    if token:
        try:
            from huggingface_hub import HfApi
            api = HfApi(token=token)
            repo_id = f"{api.whoami()['name']}/{HF_REPO_NAME}"
            api.create_repo(repo_id, repo_type="model", private=True, exist_ok=True)
            api.upload_folder(folder_path=str(OUT), repo_id=repo_id, repo_type="model",
                              commit_message=f"SafeCrop app model files {STAMP}")
            HF_URL = f"https://huggingface.co/{repo_id}"
            private = api.model_info(repo_id).private
            print("Uploaded to", HF_URL, "(private)" if private else "-- WARNING: this repo is PUBLIC, make it private")
        except Exception as e:
            print("Hugging Face upload failed:", type(e).__name__, str(e).replace(token, "***")[:300])
            print("Nothing is lost: the files are in Google Drive, and Claude can upload them later.")
    token = None
elif SMOKE_TEST:
    print("(smoke test: no Hugging Face upload)")
done()
print(f"\n══════════ ALL DONE in {(time.time() - T_START) / 60:.0f} minutes ══════════")
print(f"Google Drive: {DRIVE_OUT or OUT}  (+ {zip_path.name})")
if HF_URL:
    print("Hugging Face:", HF_URL)
print(f"Cut-offs: sure >= {SURE}, not sure < {NOT_SURE}, not a leaf < {NOT_A_LEAF}, resample: {USE_RESAMPLE}")
print("Keep this tab open a minute so Google Drive finishes saving. Then tell Claude Code 'done'.")

## Next
Tell Claude Code **done**. It puts these files into `backend/models/`, writes `docs/MODEL_CARD.md` from `README.md`,
and pins scikit-learn in `backend/requirements.txt`. If a cell failed, copy its red error text to Claude.